# Week 2 Day 3 — Reference Notebook

Topics covered:

1. **Multi-model agents** — OpenAI-compatible endpoints (Gemini, OpenRouter, Groq)
2. **Agent-as-tool orchestration** — manager agent calling specialist agents
3. **Structured outputs** — Pydantic models via `output_type`
4. **Output guardrails** — block / tripwire on bad agent output
5. **Sandbox agents** (optional) — local filesystem workspace + bug fix
6. **MCP teaser** (optional) — connect an external MCP server

Run cells top-to-bottom. Requires API keys in `.env` for the providers you use.

In [1]:
# --- Core imports for OpenAI Agents SDK workflows ---
from dotenv import load_dotenv          # Load secrets from .env into os.environ
from openai import AsyncOpenAI          # Async client; also works with non-OpenAI OpenAI-compatible APIs
from agents import (
    Agent,                              # Define an LLM agent (instructions + model + tools/guardrails)
    Runner,                             # Execute an agent: await Runner.run(agent, input)
    trace,                              # Context manager that groups a run in the Agents trace UI
    function_tool,                      # Decorator: turn a Python function into an agent tool
    OpenAIChatCompletionsModel,         # Wrap any OpenAI-compatible chat-completions endpoint as a model
    output_guardrail,                   # Decorator: validate/block agent final output
    GuardrailFunctionOutput,            # Return type for guardrail functions (tripwire + info)
)
import os
from pydantic import BaseModel, Field   # Structured outputs: schema the model must fill

In [2]:
# override=True: .env values win over any already-set shell env vars
load_dotenv(override=True)

True

In [3]:
# Read provider keys from the environment (loaded from .env above).
# OpenAI is used by the manager / checker; the others power the multi-model sales writers.
openai_api_key = os.getenv("OPENAI_API_KEY")
google_api_key = os.getenv("GOOGLE_API_KEY")          # Gemini
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")  # e.g. Kimi via OpenRouter
groq_api_key = os.getenv("GROQ_API_KEY")              # e.g. GPT-OSS on Groq

# Sanity-check that keys loaded (print only a short prefix — never log full secrets).
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set")

if openrouter_api_key:
    print(f"OpenRouter API Key exists and begins {openrouter_api_key[:6]}")
else:
    print("OpenRouter API Key not set")

if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:4]}")
else:
    print("Groq API Key not set")

OpenAI API Key exists and begins sk-proj-
Google API Key exists and begins AQ
OpenRouter API Key exists and begins sk-or-
Groq API Key exists and begins gsk_


In [4]:
# Shared system prompt for all three sales-writer agents.
# Same instructions + different models → useful diversity for the manager to pick from.
sales_instructions = """
You are a sales agent working for ComplAI, a company that
provides a SaaS tool for ensuring SOC2 compliance and preparing
for audits, powered by AI. You write compelling sales emails that
are likely to get a response.
"""

## Part 1: Multi-model agents via OpenAI-compatible endpoints

Pattern (3 steps):

1. Find the provider's **OpenAI-compatible base URL**
2. Create an `AsyncOpenAI` client pointed at that URL + API key
3. Wrap it in `OpenAIChatCompletionsModel`, then pass that as `model=` to `Agent`

#### Step 1: Base URLs

In [5]:
# Each provider exposes a chat-completions API shaped like OpenAI's.
# Point AsyncOpenAI(base_url=...) at these instead of api.openai.com.
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
GROQ_BASE_URL = "https://api.groq.com/openai/v1"

#### Step 2: Create an `AsyncOpenAI` client per provider

Same client class as OpenAI — only `base_url` and `api_key` change.

In [6]:
# One async client per provider. Agents SDK will call these under the hood.
gemini_client = AsyncOpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)
openrouter_client = AsyncOpenAI(base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key)
groq_client = AsyncOpenAI(base_url=GROQ_BASE_URL, api_key=groq_api_key)

#### Step 3: Wrap each client in `OpenAIChatCompletionsModel`

`model=` is the provider-specific model id string. That object becomes the Agent's `model`.

In [7]:
# Bind (model id + client) into Agents SDK model objects.
# Model id strings are provider-specific — check docs if one 404s / is renamed.
gemini_model = OpenAIChatCompletionsModel(
    model="gemini-3.1-flash-lite",
    openai_client=gemini_client,
)
kimi_model = OpenAIChatCompletionsModel(
    model="moonshotai/kimi-k2.6",          # OpenRouter uses org/model style ids
    openai_client=openrouter_client,
)
oss_model = OpenAIChatCompletionsModel(
    model="openai/gpt-oss-120b",
    openai_client=groq_client,
)

In [8]:
# Three specialist agents: identical instructions, different underlying models.
# Later we expose each as a tool so a manager can call them independently.
sales_agent1 = Agent(
    name="Gemini Sales Agent",
    instructions=sales_instructions,
    model=gemini_model,
)
sales_agent2 = Agent(
    name="Kimi Sales Agent",
    instructions=sales_instructions,
    model=kimi_model,
)
sales_agent3 = Agent(
    name="GPT-OSS Sales Agent",
    instructions=sales_instructions,
    model=oss_model,
)

In [9]:
# Agent.as_tool() turns an Agent into a FunctionTool another agent can call.
# The parent sees tool_name + tool_description; invoking the tool runs that agent.
# tip: keep the description action-oriented so the manager knows when to call it.
description = """Use this tool to write a sales email. In the
input, just instruct it to write a sales email.
"""

tool1 = sales_agent1.as_tool(tool_name="sales_agent1", tool_description=description)
tool2 = sales_agent2.as_tool(tool_name="sales_agent2", tool_description=description)
tool3 = sales_agent3.as_tool(tool_name="sales_agent3", tool_description=description)

In [10]:
import sys
from pathlib import Path

# This notebook lives in my-work/; course helper messenger.py is one level up (2_openai/).
# Adding that parent to sys.path lets us import without copying the file.
sys.path.insert(0, str(Path("..").resolve()))

from messenger import send_email, push

# Toggle delivery channel without changing the tool signature.
USE_EMAIL = True  # False → push notification instead of email


def send_message(subject, text_body, html_body):
    """Thin wrapper so tools / tests can switch email vs push in one place."""
    if USE_EMAIL:
        send_email(subject, text_body, html_body)
    else:
        push(f"Subject: {subject}\n\n{text_body}")

In [12]:
# Smoke-test delivery before wiring it into an agent tool.
send_message(
    "Yet another test",
    "Hooray!",
    "<html><body><h1>Hooray!</h1></body></html>",
)

In [13]:
# @function_tool exposes a normal Python function to the LLM as a tool.
# The docstring + type hints become the tool schema the model uses for arguments.
@function_tool
def send_email_tool(subject: str, text_body: str, html_body: str) -> str:
    """
    Send out an email with the given subject and body to all sales prospects.

    Args:
        subject: The subject of the email
        text_body: The body of the email as plain text
        html_body: The HTML body of the email
    """
    send_message(subject, text_body, html_body)
    return "Email sent successfully"  # Short confirmation string returned to the agent

In [14]:
# Toolbelt for the sales manager:
# - 3 agent-as-tools → generate draft emails from different models
# - 1 function tool → send the chosen email
tools = [tool1, tool2, tool3, send_email_tool]

In [15]:
# Manager agent: orchestrates specialists via tools (LLM decides call order).
# model="gpt-5.4-mini" uses the default OpenAI client (OPENAI_API_KEY) — a plain string
# model id, not an OpenAIChatCompletionsModel wrapper.
manager_instructions = """
You are a Sales Manager at ComplAI. Your goal is to find the
single best cold sales email using the sales_writer tools.
"""

# Explicit multi-step task reduces skipped tools / double-sends.
task = """
Follow these steps:

1. Generate Drafts: Use each of the three sales_email_writer
tools to generate different email drafts. Just instruct each
to write a sales email; no further details are needed. Do not
proceed until all three drafts are ready, one from each tool.

2. Evaluate and Select: Review the drafts and choose the single
best email using your judgement of which one is most effective.

3. Use your tool to send the best email (and only the best email)
to the user, Only send 1 email.
"""

sales_manager = Agent(
    name="Sales Manager",
    instructions=manager_instructions,
    tools=tools,
    model="gpt-5.4-mini",
)

In [16]:
# trace(...) names this run in the Agents dashboard / local traces for debugging.
# Runner.run is async — must await (notebooks support top-level await).
with trace("Sales Manager across different models"):
    result = await Runner.run(sales_manager, task)

# final_output is the agent's last message after tools finish.
print(result.final_output)

Done — I generated three drafts, selected the strongest one, and sent only that email.


## Part 2: Structured Outputs

Instead of free-text, force the agent to return a **Pydantic model**.

Pattern:
1. Define a `BaseModel` with `Field(description=...)` — descriptions guide the LLM
2. Pass `output_type=YourModel` when constructing the Agent
3. `result.final_output` is a typed instance (not a string)

Useful for validation, routing, and feeding guardrails.

In [17]:
# Schema the checker agent must fill. Field descriptions become part of the JSON schema
# sent to the model — write them carefully; they are the "prompt" for each field.
class EmailReview(BaseModel):
    is_professional: bool = Field(
        description="Whether the email is professional and appropriate"
    )
    number_of_sentences: int = Field(
        description=(
            "The number of sentences in the body of the email, "
            "not including the greeting and signature"
        )
    )
    contains_placeholders: bool = Field(
        description="Whether the email contains placeholders for personalization"
    ) 

In [18]:
# Inspect the JSON Schema derived from the Pydantic model (what the API sees).
EmailReview.model_json_schema()

{'properties': {'is_professional': {'description': 'Whether the email is professional and appropriate',
   'title': 'Is Professional',
   'type': 'boolean'},
  'number_of_sentences': {'description': 'The number of sentences in the body of the email, not including the greeting and signature',
   'title': 'Number Of Sentences',
   'type': 'integer'},
  'contains_placeholders': {'description': 'Whether the email contains placeholders for personalization',
   'title': 'Contains Placeholders',
   'type': 'boolean'}},
 'required': ['is_professional',
  'number_of_sentences',
  'contains_placeholders'],
 'title': 'EmailReview',
 'type': 'object'}

In [19]:
# Deliberately bad sample: informal tone + [first_name] placeholder.
# Expect: is_professional=False, contains_placeholders=True.
email = """
Hi [first_name],

I'm hitting you up to see if you'd like to buy our product. It's
really great. You'll miss out if you don't buy it.

Laters.

Michael
"""

In [20]:
# output_type=EmailReview → Runner returns a parsed EmailReview, not raw text.
checker = Agent(
    name="Checker",
    instructions="You review potential sales emails",
    model="gpt-5.4-mini",
    output_type=EmailReview,
)
result = await Runner.run(checker, email)

In [21]:
# Typed object — attribute access works; no JSON.loads needed.
review = result.final_output
review

EmailReview(is_professional=False, number_of_sentences=3, contains_placeholders=True)

In [22]:
# Example field access for branching logic / guardrails.
review.is_professional

False

## Part 3: Output Guardrails

Guardrails run **after** the agent produces a final answer.

- Decorate an async function with `@output_guardrail`
- Return `GuardrailFunctionOutput(tripwire_triggered=...)`
- If `tripwire_triggered=True`, `Runner.run` raises `OutputGuardrailTripwireTriggered`

Here the checker (structured output) is reused inside the guardrail to decide pass/fail.

In [23]:
# Output guardrail signature: (ctx, agent, message) → GuardrailFunctionOutput
# message is the agent's proposed final output (string or structured, depending on agent).
@output_guardrail
async def email_guardrail(ctx, agent, message):
    # Reuse the structured checker agent to score the draft.
    result = await Runner.run(checker, message, context=ctx.context)
    review = result.final_output

    # Tripwire if placeholders remain OR tone is unprofessional.
    is_problem = review.contains_placeholders or not review.is_professional

    return GuardrailFunctionOutput(
        output_info={"review": review},  # optional debug payload
        tripwire_triggered=is_problem,   # True → run raises OutputGuardrailTripwireTriggered
    ) 

In [24]:
# Cowboy persona often triggers the guardrail (informal / placeholders).
# Attach guardrails on the Agent via output_guardrails=[...].
cowboy_instructions = sales_instructions + "\nSpeak like a cowboy"

sales_agent_cowboy = Agent(
    name="Cowboy",
    instructions=cowboy_instructions,
    model=gemini_model,
    output_guardrails=[email_guardrail],
)

In [25]:
# Expected: OutputGuardrailTripwireTriggered if the cowboy email fails the review.
# Wrap in try/except in production; here we let it surface to show the failure mode.
result = await Runner.run(sales_agent_cowboy, "Write a cold sales email")
result.final_output

OutputGuardrailTripwireTriggered: Guardrail OutputGuardrail triggered tripwire

In [26]:
# Same cowboy agent WITHOUT guardrails — run succeeds and we capture the email text
# so we can inspect why the guardrail would have fired.
simple_cowboy = Agent(
    name="Simple Cowboy",
    instructions=cowboy_instructions,
    model=gemini_model,
)
result = await Runner.run(simple_cowboy, "Write a cold sales email")
email = result.final_output
print(email)

Subject: Wranglin’ your SOC2 compliance before the auditor rides into town

Howdy, [Prospect Name],

I reckon you’ve been spendin’ more time tanglin’ with spreadsheets and documentation than actually buildin’ the product your customers are clamorin’ for. I’ve seen many a fine outfit get bucked off their horse when audit season rolls around—it’s a dusty, miserable trail to ride alone.

I’m writin’ to you from ComplAI. We’ve built a tool that takes the back-breakin’ labor out of gettin’ your SOC2 compliance squared away. Think of us as your personal trail boss—our AI gathers the evidence, spots the gaps, and keeps your posture sharp so you don’t get blindsided when the auditors come knockin’.

We help teams cut their prep time down to a fraction, so you can spend your days buildin’ instead of fixin’ paperwork.

I’d be obliged if you had ten minutes this week to chew the fat and see if we can save you some sweat. You free for a quick call this Thursday?

Best trails,

[Your Name]
Sales Ag

In [27]:
# Manual equivalent of the guardrail decision (useful for debugging without exceptions).
result = await Runner.run(checker, email)
review = result.final_output

if not review.is_professional or review.contains_placeholders:
    print("The email is not professional or has placeholders and will not be sent")
else:
    print("Email is good") 

The email is not professional or has placeholders and will not be sent


## Optional EXTRA: Sandbox Agents

Sandbox agents get a **workspace** (files + shell) instead of only chat/tools.

Key pieces:
| Piece | Role |
|---|---|
| `Manifest` | What gets copied into the sandbox (e.g. `LocalDir`) |
| `SandboxPathGrant` | Extra host paths the agent may write to |
| `Capabilities` | Tools enabled (filesystem, shell, …) |
| `UnixLocalSandboxClient` | Runs the sandbox on this machine (temp dir) |
| `SandboxAgent` | Agent subclass wired to that workspace |

**Path note (this notebook):** course sample code lives at `../code` (sibling of `my-work/`), not `./code`.

Docs: https://openai.github.io/openai-agents-python/sandbox_agents/

In [28]:
# Sandbox-specific imports (separate from the core Agent/Runner used above).
from pathlib import Path
from agents.run import RunConfig
from agents.sandbox import Manifest, SandboxAgent, SandboxRunConfig, SandboxPathGrant
from agents.sandbox.capabilities import Capabilities
from agents.sandbox.entries import LocalDir
from agents.sandbox.sandboxes.unix_local import UnixLocalSandboxClient

In [29]:
# Host paths for the sandbox manifest.
# CODE_DIR: buggy sample (orders.py) lives in the course folder one level up.
# OUTPUT_DIR: local folder the agent may write fixed files into (via path grant).
#
# If this notebook were in agents/2_openai/ (not my-work/), you'd use Path("code").
CODE_DIR = Path("..").joinpath("code").resolve()
OUTPUT_DIR = Path("output").resolve()
if not OUTPUT_DIR.exists():
    OUTPUT_DIR.mkdir() 

In [30]:
# Confirm CODE_DIR points at .../agents/2_openai/code (contains orders.py).
CODE_DIR

PosixPath('/Users/michaelperrine/Documents/Donner/ai-engineer-agentic-track-donner/agents/2_openai/code')

In [31]:
# Tell the engineer where to write fixes. Files inside the sandbox appear under
# the manifest key (here: code/...), so patches should target code/orders.py —
# not bare orders.py at the sandbox root.
instructions = f"""
You are a software engineer that fixes bugs.
Review files in the sandbox code directory.

Write the fixed version of the file to this host output directory.
{OUTPUT_DIR}

Use full file paths when writing output.
Respond with a summary of what you did.
"""

In [34]:
# Manifest materializes host files into the sandbox workspace.
# entries={"code": LocalDir(...)} → sandbox path "code/" mirrors CODE_DIR.
# extra_path_grants → allow writing outside the ephemeral workspace (OUTPUT_DIR).
manifest = Manifest(
    entries={"code": LocalDir(src=CODE_DIR)},
    extra_path_grants=[SandboxPathGrant(path=str(OUTPUT_DIR))],
)

# Default capabilities: filesystem + shell (+ compaction). Enough for edit/run loops.
capabilities = Capabilities.default()
capabilities

[Filesystem(type='filesystem', session=None, run_as=None, configure_tools=None),
 Shell(type='shell', session=None, run_as=None, configure_tools=None),
 Compaction(type='compaction', session=None, run_as=None, policy=None)]

In [32]:
# RunConfig.sandbox tells Runner which sandbox backend to use for SandboxAgent.
# UnixLocalSandboxClient → temp dir on this machine (no remote VM).
run_config = RunConfig(
    sandbox=SandboxRunConfig(client=UnixLocalSandboxClient()),
    workflow_name="Sandbox coding example",
)

In [35]:
# SandboxAgent = Agent + default_manifest + capabilities.
# On start, the SDK copies LocalDir into the session and exposes FS/shell tools.
agent = SandboxAgent(
    name="Engineer",
    instructions=instructions,
    model="gpt-5.4-mini",
    default_manifest=manifest,
    capabilities=capabilities,
)

In [36]:
# Bug in orders.py: dict.fromkeys(..., []) shares one list across all keys.
# Agent should fix that and write output/orders.py.
#
# Note: tool errors mid-run (e.g. apply_patch on wrong path) can appear in the
# printed stream; if final_output summarizes a real fix and output/ exists, OK.
result = await Runner.run(agent, "Fix the bug in the code", run_config=run_config)
print(result.final_output)

Fixed the bug in `code/orders.py` and copied the corrected file to `/Users/michaelperrine/Documents/Donner/ai-engineer-agentic-track-donner/agents/2_openai/my-work/output/orders.py`.

What changed:
- Replaced `dict.fromkeys(..., [])`, which caused every customer to share the same list.
- Built the grouped dictionary with a fresh list per `customer_id` using a dict comprehension.

If you want, I can also add a small test for this function.


## Optional EXTRA: MCP Teaser

**MCP** (Model Context Protocol) lets an agent call tools from an external server
(docs search, DBs, etc.) without you hand-writing each tool.

Pattern:
1. Open an MCP client connection (`async with MCPServerStreamableHttp(...)`)
2. Pass `mcp_servers=[server]` to `Agent`
3. The SDK discovers remote tools and the model can call them during `Runner.run`

In [37]:
# Streamable HTTP transport: connect to a remote MCP server over the network.
from agents.mcp import MCPServerStreamableHttp

In [38]:
# Prompt designed so the agent should use Context7 (docs MCP) rather than guess.
task = """
In the new SandboxAgents feature in the OpenAI Agents SDK as of
May 2026, what is the role of the Manifest object?
Always be accurate. If you don't know the answer, say so.
"""

In [39]:
# params["url"] = MCP endpoint; timeout in seconds for the HTTP session.
params = {"url": "https://mcp.context7.com/mcp", "timeout": 60}

# async with ensures the MCP session is opened and cleanly closed.
# mcp_servers=[server] registers remote tools on this Agent for the run.
async with MCPServerStreamableHttp(name="Context7", params=params) as server:
    agent = Agent(
        name="Expert",
        instructions="Use Context7 to answer the question",
        mcp_servers=[server],
        model="gpt-4o-mini",
    )
    result = await Runner.run(agent, task)

print(result.final_output)

In the new **SandboxAgents** feature of the OpenAI Agents SDK, the **Manifest object** plays a crucial role in configuring the sandbox environment for agents. Its primary functions include:

1. **Defining Entries**: The Manifest allows you to declare local directories and other resources that an agent can access during its operations. This includes setting paths for input data and output directories.

2. **User Permissions**: It can define specific users and their permissions to control access to various resources. For example, you can specify which users can read, write, or execute files in the sandbox.

3. **Path Grants**: It supports extra path grants for the sandbox environment, ensuring that certain directory paths are accessible to the agent while maintaining configuration security.

4. **Skill Integration**: The Manifest can also be used to include custom skills that the agent can utilize to perform specific tasks, like accessing tools or executing commands.

Overall, the Manife